# SMS Spam Detection Using LSTM
## Exploratory Data Analysis & Dataset Validation
**Dataset Source**: Official UCI Machine Learning Repository (SMS Spam Collection Dataset)  
**Reference Citation**:
> Almeida, T.A., Gómez Hidalgo, J.M., Yamakami, A. (2011). Contributions to the study of SMS Spam Filtering: New Collection and Results. *Proceedings of the 2011 ACM Symposium on Document Engineering (ACM DOCENG'11)*, Mountain View, CA, USA.



### 1. Environment Setup & Library Imports


In [ ]:
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import nltk
from nltk.corpus import stopwords

# Ensure required NLTK corpora are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Styling
sns.set_theme(style='whitegrid', font='sans-serif')
colors = {'ham': '#2b5c8f', 'spam': '#d9534f'}



### 2. Dataset Acquisition & Raw Data Ingestion
We load the original tab-separated dataset from `data/raw/SMSSpamCollection` while preserving the raw source files unchanged.
The pipeline also supports Kaggle-style CSV formatting.


In [ ]:
# Resolve paths
DATA_PATH = Path('../data/raw/SMSSpamCollection')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/raw/SMSSpamCollection')

print(f'Loading raw dataset from: {DATA_PATH.resolve()}')

# Tab-separated without header
raw_df = pd.read_csv(
    DATA_PATH,
    sep='\t',
    header=None,
    names=['raw_label', 'raw_message'],
    encoding='utf-8'
)

print(f'Raw dataset shape: {raw_df.shape}')
raw_df.head()



### 3. Normalization into a Working DataFrame
We create a separate working DataFrame to standardize column names (`message`, `label`, `label_encoded`) and convert labels into binary indicators:
- `ham` $\rightarrow$ `0` (Legitimate SMS)
- `spam` $\rightarrow$ `1` (Unsolicited / Fraudulent SMS)



In [ ]:
working_df = pd.DataFrame()
working_df['message'] = raw_df['raw_message'].astype(str)
working_df['label'] = raw_df['raw_label'].astype(str).str.lower().str.strip()

label_map = {'ham': 0, 'spam': 1}
working_df['label_encoded'] = working_df['label'].map(label_map)

print('Working DataFrame Info:')
print(working_df.info())
working_df.head()



### 4. Data Quality, Missing Values, and Duplicate Checks


In [ ]:
total_records = len(working_df)
missing_messages = working_df['message'].isna().sum() + (working_df['message'].str.strip() == '').sum()
missing_labels = working_df['label'].isna().sum()
invalid_labels = working_df['label_encoded'].isna().sum()

duplicate_rows = working_df.duplicated().sum()
duplicate_messages = working_df.duplicated(subset=['message']).sum()
unique_messages = working_df['message'].nunique()

# Check for conflicting labels across duplicate messages
conflicts = working_df.groupby('message')['label_encoded'].nunique()
conflicting_messages = (conflicts > 1).sum()

print('=== DATA VALIDATION AUDIT ===')
print(f'Total Records:            {total_records}')
print(f'Missing Messages:         {missing_messages}')
print(f'Missing Labels:           {missing_labels}')
print(f'Invalid/Unmapped Labels:  {invalid_labels}')
print(f'Duplicate Message Texts:  {duplicate_messages}')
print(f'Unique Message Texts:     {unique_messages}')
print(f'Conflicting Label Texts:  {conflicting_messages}')



### 5. Class Distribution & Imbalance Analysis


In [ ]:
class_counts = working_df['label'].value_counts()
class_pcts = working_df['label'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pcts.round(2)
})
print(summary_df)

# Visualize Class Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = ax1.bar(['Ham (0)', 'Spam (1)'], class_counts.values, color=[colors['ham'], colors['spam']], width=0.5, edgecolor='black', linewidth=1.2)
ax1.set_title('SMS Spam vs Ham Class Counts', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Number of Messages')
ax1.set_ylim(0, 5500)
for bar, count, pct in zip(bars, class_counts.values, class_pcts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{count:,} ({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Donut chart
ax2.pie(class_counts.values, labels=['Ham', 'Spam'], autopct='%1.1f%%', startangle=140, colors=[colors['ham'], colors['spam']], explode=(0, 0.08), wedgeprops=dict(width=0.45, edgecolor='black', linewidth=1.2))
ax2.set_title('Class Proportion (86.6% Ham vs 13.4% Spam)', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()



### 6. Message Length & Linguistic Statistics
We calculate character counts, word counts, and average word lengths for each message to uncover behavioral differences between spam and ham.



In [ ]:
working_df['char_count'] = working_df['message'].apply(len)
working_df['word_count'] = working_df['message'].apply(lambda x: len(x.split()))
working_df['avg_word_length'] = working_df.apply(
    lambda r: (r['char_count'] / r['word_count']) if r['word_count'] > 0 else 0.0,
    axis=1
)

print('--- Character Count Summary by Class ---')
display_char = working_df.groupby('label')['char_count'].describe().round(2)
print(display_char)

print('\n--- Word Count Summary by Class ---')
display_word = working_df.groupby('label')['word_count'].describe().round(2)
print(display_word)

# Visualization: Message Length Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(data=working_df, x='char_count', hue='label', bins=50, kde=True, palette=colors, ax=ax1, stat='density', common_norm=False, alpha=0.4)
ax1.set_title('Character Length Distribution (KDE)', fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Character Count')
ax1.set_xlim(0, 300)
ax1.axvline(160, color='gray', linestyle='--', linewidth=1.2, label='160-char SMS Limit')
ax1.legend(['Ham', 'Spam', '160-char Limit'])

sns.boxplot(data=working_df, x='label', y='word_count', palette=colors, ax=ax2, width=0.4)
ax2.set_title('Word Count Comparison (Boxplot)', fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Label')
ax2.set_ylabel('Word Count')
ax2.set_ylim(0, 60)

plt.tight_layout()
plt.show()



### 7. Word Frequency Analysis (Top Keywords)
Extracting the most frequent tokens for both classes (filtering out common English stopwords and HTML remnants like `&lt;` and `&gt;`).


In [ ]:
stop_words = set(stopwords.words('english'))
html_artifacts = {'lt', 'gt', 'amp'}

def get_top_tokens(texts, n=20):
    tokens = []
    for t in texts:
        words = re.findall(r'[a-zA-Z]{2,}', t.lower())
        tokens.extend([w for w in words if w not in stop_words and w not in html_artifacts])
    return pd.DataFrame(Counter(tokens).most_common(n), columns=['Word', 'Count'])

spam_keywords = get_top_tokens(working_df[working_df['label'] == 'spam']['message'], 20)
ham_keywords = get_top_tokens(working_df[working_df['label'] == 'ham']['message'], 20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=spam_keywords, y='Word', x='Count', color=colors['spam'], ax=ax1, edgecolor='black', linewidth=0.8)
ax1.set_title('Top 20 Keywords in SPAM Messages', fontsize=12, fontweight='bold', pad=10)
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_width())}', (p.get_width() + 3, p.get_y() + p.get_height()/2), va='center', fontsize=9)

sns.barplot(data=ham_keywords, y='Word', x='Count', color=colors['ham'], ax=ax2, edgecolor='black', linewidth=0.8)
ax2.set_title('Top 20 Keywords in HAM Messages', fontsize=12, fontweight='bold', pad=10)
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_width())}', (p.get_width() + 3, p.get_y() + p.get_height()/2), va='center', fontsize=9)

plt.tight_layout()
plt.show()



### 8. Representative Examples


In [ ]:
print('=== 5 REPRESENTATIVE HAM EXAMPLES ===')
for idx, text in working_df[working_df['label'] == 'ham'].head(5)['message'].items():
    print(f'HAM #{idx} [{len(text)} chars | {len(text.split())} words]:\n  "{text}"\n')

print('=== 5 REPRESENTATIVE SPAM EXAMPLES ===')
for idx, text in working_df[working_df['label'] == 'spam'].head(5)['message'].items():
    print(f'SPAM #{idx} [{len(text)} chars | {len(text.split())} words]:\n  "{text}"\n')



### 9. Key Insights & LSTM Modeling Considerations

1. **Severe Class Imbalance**: 86.6% Ham vs. 13.4% Spam.
   - *Impact*: Raw accuracy is misleading (a naive classifier achieves 86.6%).
   - *Mitigation*: We must evaluate using **Precision, Recall, F1-Score, and ROC-AUC**, and configure class weights (`compute_class_weight`) during model training.

2. **Distinct Message Length Signatures**:
   - Spam messages have a median character count of **149** (mean 138.7, tightly clustered near the 160-character SMS ceiling).
   - Ham messages have a median character count of **52** (mean 71.5, heavily right-skewed).
   - *Sequence Length*: A maximum sequence length (`max_len`) of **100 words** covers 99.5% of all messages without truncation.

3. **Spam Keyword Density**:
   - Spam is dominated by urgency triggers, financial tokens, and call-to-actions: `call`, `free`, `txt`, `claim`, `prize`, `won`, `urgent`, `cash`.
   - Ham messages are colloquial and conversational: `ok`, `go`, `good`, `come`, `love`, `day`, `home`.

4. **Hinglish Exploration Pathway**:
   - Hinglish messages introduce code-switching, Romanized Hindi phonetic variants (*paisa / paise*, *karo / kijiye*), and informal SMS shorthand.
   - Sub-word tokenization and custom slang dictionaries in `research/` will be explored to prevent high out-of-vocabulary (OOV) rates.

